In [ ]:
import sys
if 'google.colab' in sys.modules:
    from IPython.core.getipython import get_ipython
    get_ipython().run_line_magic("pip", "install transformers sentencepiece accelerate")

In [ ]:
import torch
import activation_additions as aa


import csv

import scipy.stats as stats

from typing import Dict, Union, Callable, List
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector
import matplotlib.pyplot as plt


from nltk.tokenize import word_tokenize
from tqdm.notebook import tqdm


from datasets import load_dataset

import torch.nn.functional as F
from collections import defaultdict

import pickle
from collections import Counter
import random
import pandas as pd

In [ ]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

In [ ]:
MODEL = "openai-community/gpt2-xl"
# We use 16 bits percision to save time
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
NUM_BLOCKS = len(aa.get_blocks(model))

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "tokens_to_generate": 50,
    "seed": 0, 
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

#### Load the dataset (run only once!)

In [ ]:
local_pool = load_dataset("openwebtext", split="train[:200000]")

In [ ]:
print(len(local_pool))

#### Preproccess the dataset and save in a file (run only once)

In [ ]:
processed_documents = []

print("Processing documents...")
for doc in tqdm(local_pool["text"]):
    # 1. Tokenize, truncate/pad to exactly 1024 tokens
    encoded = tokenizer(
        doc, 
        truncation=True, 
        max_length=1024, 
        padding="max_length", 
        add_special_tokens=False
    )
    
    # Decode back into a string
    trimmed_string = tokenizer.decode(encoded["input_ids"])
    
    # 2. Extract words and count appearances using NLTK
    # Lowercasing to ensure reliable keyword matching later
    words = word_tokenize(trimmed_string.lower())
    word_counts = Counter(words)
    
    # Storing both the string and its frequency map so the final function can return the text
    processed_documents.append({
        "text": trimmed_string,
        "counts": word_counts
    })

# 3. Pickle the list
with open("processed_documents.pkl", "wb") as f:
    pickle.dump(processed_documents, f)

print("Saved processed documents to processed_documents.pkl")

#### Load the preproccesed dataset

In [ ]:
pkl_path="processed_documents.pkl"
print("Loading prepoccessed data")
with open(pkl_path, "rb") as f:
    documents = pickle.load(f)
print("Shuffeling data...")
random.shuffle(documents)
print(f"Done: loaded {len(documents)} documents")

### The Expiriment

In [ ]:
def get_documents(words, document_count, min_concept_count=55, keyword_min_appearances=3, csv_path="key_words.csv"):
    """
    Fetches documents related to a list of words, aiming for an equal split, 
    but allowing uneven distributions as long as each word meets the min_concept_count.
    """
    # 0. Sanity Check
    if min_concept_count * len(words) > document_count:
        raise ValueError(
            f"Impossible configuration: {len(words)} words with a minimum of {min_concept_count} "
            f"requires at least {min_concept_count * len(words)} documents, but you only asked for {document_count}."
        )

        
    df = pd.read_csv(csv_path)
    
    # 1. Map each target word to its list of keywords
    word_to_keywords = {}
    for word in words:
        row = df[df['Hypothesis'] == word]
        if row.empty:
            raise ValueError(f"Word '{word}' not found in {csv_path}")
        raw_keywords = str(row.iloc[0]['Keywords'])
        word_to_keywords[word] = [kw.strip().lower() for kw in raw_keywords.split(',')]
        
    
    # Track documents in buckets for easy balancing
    related_buckets = {word: [] for word in words}
    unrelated = []
    
    # Helper to evaluate a single document
    def is_related(doc, keywords):
        counts = {str(k).lower(): v for k, v in doc["counts"].items()}
        total_words = sum(counts.values())
        if total_words == 0:
            return False
        # total_occurrences = sum(counts[kw] for kw in keywords if kw in counts)
        split_keywords = [kw.split() for kw in keywords]        
        total_occurrences = sum(min(counts[kkw] for kkw in kw) for kw in split_keywords if all(kkw in counts for kkw in kw))
        
        
        return total_occurrences >= keyword_min_appearances

    # 2. Iterate and route the documents
    for doc in documents:
        unrelated_full = len(unrelated) >= document_count
        related_total = sum(len(l) for l in related_buckets.values())
        mins_met = all(len(l) >= min_concept_count for l in related_buckets.values())
        
        # We can stop looking at related docs if we hit the total target AND all minimums are met
        related_full = (related_total >= document_count) and mins_met
        
        if unrelated_full and related_full:
            break
            
        # Find all target words this document relates to
        matched_words = [word for word in words if is_related(doc, word_to_keywords[word])]
                
        if not matched_words:
            if not unrelated_full:
                unrelated.append(doc["text"])
        else:
            if not related_full:
                # To balance the dataset, assign this doc to the matched word that currently has the FEWEST documents
                best_word = min(matched_words, key=lambda w: len(related_buckets[w]))
                
                # If we already have enough total docs, ONLY add if this specific word is starving
                if related_total >= document_count:
                    if len(related_buckets[best_word]) < min_concept_count:
                        related_buckets[best_word].append(doc["text"])
                else:
                    related_buckets[best_word].append(doc["text"])

    # 3. Validation Checks
    for word in words:
        if len(related_buckets[word]) < min_concept_count:
            raise ValueError(
                f"Not enough documents found for '{word}'. "
                f"Needed at least {min_concept_count}, but only found {len(related_buckets[word])}."
            )
            
        
    if len(unrelated) < document_count:
        raise ValueError(f"Not enough unrelated documents found. Needed {document_count}, but only found {len(unrelated)}.")

    # 4. Trim Excess (If we over-collected to rescue a starving word, shave off the surplus)
    while sum(len(l) for l in related_buckets.values()) > document_count:
        largest_bucket = max(words, key=lambda w: len(related_buckets[w]))
        related_buckets[largest_bucket].pop()

    # 5. Merge and return
    related = []
    for word in words:
        related.extend(related_buckets[word])
    
    # Print how much was find for each hypotesys
    for l in related_buckets:
        print(f"Found {len(related_buckets[l])} for {l}")
        
    return related + unrelated

In [ ]:
def get_logprob_shifts(
    concept_names: List[str],
    pos_prompts: List[str],
    neg_prompts: List[str],
    hypotheses: List[str],
    layers: List[int],
    coeffs: List[float],
    num_docs: int = 500,
    batch_size: int = 4,
    token_min_appearances: int = 10,
    max_length: int = 1024,
    min_doc_keywords= 3,
    keywords_csv_path="data_hub\log_prob_shift_keywords.csv",
) -> List[Dict]:
    """
    Evaluates logprob shifts across documents for multiple simultaneous concepts in batches.
    """
    assert len(pos_prompts) == len(neg_prompts) == len(layers) == len(coeffs) == len(concept_names), \
        "All input lists must be of the same length."

    # SETUP HOOKS
    print("1. Generating Activation Addition hooks...")
    all_additions = []
    for concept, p_add, p_sub, layer, coeff in zip(concept_names, pos_prompts, neg_prompts, layers, coeffs):
        additions = get_x_vector_preset(
            prompt1=p_add, 
            prompt2=p_sub, 
            coeff=coeff, 
            act_name=layer
        )
        all_additions.extend(additions)

    blocks = aa.get_blocks(model)
    hooks = [(blocks[a.layer], aa.get_hook_fn(a.coeff * a.act)) for a in all_additions]

    # Maps token: sum log log_probs 
    base_logprob_sums = defaultdict(float)
    steer_logprob_sums = defaultdict(float)
    # Maps token: number of appearences 
    token_counts = defaultdict(int)

    # PROCESS DOCUMENTS
    print(f"\n2. Getting {num_docs} documents...")
   

    texts = get_documents(words=hypotheses, document_count=num_docs//2, keyword_min_appearances=min_doc_keywords, csv_path=keywords_csv_path)

    print(len(texts))

    for i in tqdm(range(0, len(texts), batch_size), desc="Evaluating Batches"):
        batch_texts = texts[i : i + batch_size]
        
        # Tokenize and truncate to max length, with padding for batching
        inputs = tokenizer(
            batch_texts, 
            return_tensors='pt', 
            padding=True, 
            truncation=True, 
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']

        # Skip if the document is too short to predict a next token
        valid_lengths = attention_mask.sum(dim=1)
        ignore_seqs = valid_lengths < 20
        attention_mask[ignore_seqs, :] = 0 
        
        if attention_mask.sum() == 0:
            continue
            
        # BASELINE FORWARD PASS
        with torch.no_grad():
            base_outputs = model(**inputs)
            base_logprobs = F.log_softmax(base_outputs.logits, dim=-1)
            
        # STEERED FORWARD PASS
        with torch.no_grad():
            with aa.pre_hooks(hooks):
                steer_outputs = model(**inputs)
                steer_logprobs = F.log_softmax(steer_outputs.logits, dim=-1)
                
        # EXTRACT PROBABILITIES
        # target_ids are the actual next tokens that appeared in the text
        target_ids = input_ids[:, 1:] 
        valid_mask = attention_mask[:, 1:] 
        
        # Gather logprobs for the actual tokens
        base_target_lps = base_logprobs[:, :-1, :].gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)
        steer_target_lps = steer_logprobs[:, :-1, :].gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)
        
        # Move to CPU immediately to free VRAM
        target_ids_cpu = target_ids.cpu()
        base_lps_cpu = base_target_lps.cpu()
        steer_lps_cpu = steer_target_lps.cpu()
        valid_mask_cpu = valid_mask.cpu()
        
        # Accumulate sums and counts ONLY for valid tokens (ignoring padding and short texts)
        for b in range(target_ids_cpu.shape[0]):
            for seq_idx in range(target_ids_cpu.shape[1]):
                if valid_mask_cpu[b, seq_idx] == 1:
                    token_id = target_ids_cpu[b, seq_idx].item()
                    base_logprob_sums[token_id] += base_lps_cpu[b, seq_idx].item()
                    steer_logprob_sums[token_id] += steer_lps_cpu[b, seq_idx].item()
                    token_counts[token_id] += 1

    # CALCULATE AVERAGES AND SHIFTS
    print("\n3. Calculating averages and filtering...")
    results = []
    for token_id, count in token_counts.items():
        if count >= token_min_appearances:
            avg_base = base_logprob_sums[token_id] / count
            avg_steer = steer_logprob_sums[token_id] / count
            shift = avg_steer - avg_base
            
            results.append({
                "token_id": token_id,
                "token_str": tokenizer.decode([token_id]),
                "appearances": count,
                "avg_base_logprob": avg_base,
                "avg_steer_logprob": avg_steer,
                "logprob_shift": shift
            })
            
    # Sort results by the largest positive shift (tokens that became much more likely)
    results.sort(key=lambda x: x["logprob_shift"], reverse=True)
    
    print(f"Finished. Kept {len(results)} unique tokens with >= {token_min_appearances} appearances.")
    return results

#### Displaying the results

In [ ]:
def plot_shift_distribution(results: List[Dict], concepts: List[str]) -> None:
    """
    Plots the distribution of logprob shifts across the vocabulary.
    X-axis: Tokens sorted by shift (labels omitted for cleanliness).
    Y-axis: Logprob shift magnitude.
    """
    if not results:
        print("No results to plot.")
        return

    # Extract just the shift values (already sorted descending by the main function)
    shifts = [res["logprob_shift"] for res in results]
    
    plt.figure(figsize=(10, 5))
    plt.plot(shifts, color='royalblue', linewidth=2)
    
    # Add a horizontal line at 0 for visual reference
    plt.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    
    concept_title = " & ".join(concepts)
    plt.title(f"Logprob Shift Distribution: {concept_title}", fontsize=14, pad=15)
    plt.ylabel("Logprob Shift (Steered - Baseline)", fontsize=12)
    plt.xlabel("Token Rank (Sorted Highest to Lowest Shift)", fontsize=12)
    
    # Grid for easier reading
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

def print_top_and_bottom_shifts(results: List[Dict], concepts: List[str], x: int = 10, y: int = 10) -> None:
    """
    Prints the top 'x' tokens with the highest positive shift 
    and the bottom 'y' tokens with the lowest negative shift.
    """
    if not results:
        print("No results to print.")
        return

    concept_title = " & ".join(concepts)
    print(f"=== Extreme Logprob Shifts for: {concept_title} ===\n")
    
    print(f"Top {x} Increased Tokens (Highest Positive Shift):")
    print("-" * 60)
    for res in results[:x]:
        # repr() safely prints spaces and newlines so formatting doesn't break
        token_safe = repr(res['token_str'])
        print(f"{token_safe:<20} | Shift: {res['logprob_shift']:>7.4f} | Appearances: {res['appearances']}")
        
    print(f"\nBottom {y} Decreased Tokens (Lowest Negative Shift):")
    print("-" * 60)
    
    # Use [-y:] to get the end of the list, and reversed() so the most negative is printed first
    bottom_results = reversed(results[-y:]) if len(results) >= y else reversed(results)
    for res in bottom_results:
        token_safe = repr(res['token_str'])
        print(f"{token_safe:<20} | Shift: {res['logprob_shift']:>7.4f} | Appearances: {res['appearances']}")
    print()

def print_specific_token_stats(results: List[Dict], target_tokens: List[str]) -> None:
    """
    Searches the results for specific token strings and prints their full stats.
    Note: GPT-2 tokens often include leading spaces (e.g., ' winter' instead of 'winter').
    """
    if not results:
        print("No results to search.")
        return
        
    # Create a quick lookup dictionary for O(1) searching
    results_dict = {res["token_str"]: res for res in results}
    
    print(f"=== Stats for Specific Tokens ===")
    print("-" * 80)
    
    for token in target_tokens:
        if token in results_dict:
            res = results_dict[token]
            token_safe = repr(res['token_str'])
            print(f"Token: {token_safe:<15} | Shift: {res['logprob_shift']:>7.4f} | "
                  f"Base: {res['avg_base_logprob']:>7.4f} | Steer: {res['avg_steer_logprob']:>7.4f} | "
                  f"Count: {res['appearances']}")
        else:
            print(f"Token: {repr(token):<15} | NOT FOUND (Check exact spelling/spaces, or it didn't hit min_appearances)")
            
    print("-" * 80)
    print()

def plot_qq_shifts(results: List[Dict], concepts: List[str]) -> None:
    """
    Creates a Q-Q plot of the logprob shifts to compare their 
    distribution against a theoretical normal distribution.
    """
    if not results:
        print("No results to plot.")
        return

    # Extract just the shift values
    shifts = [res["logprob_shift"] for res in results]
    
    plt.figure(figsize=(8, 8))
    
    # probplot automatically calculates the quantiles and plots the line of best fit
    stats.probplot(shifts, dist="norm", plot=plt)
    
    concept_title = " & ".join(concepts)
    plt.title(f"Q-Q Plot of Logprob Shifts: {concept_title}", fontsize=14, pad=15)
    plt.ylabel("Ordered Logprob Shifts", fontsize=12)
    plt.xlabel("Theoretical Quantiles (Normal Distribution)", fontsize=12)
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

#### Export and Import for Token Results

In [ ]:
def save_results_to_csv(results, combination_name):
    """Saves a list of token result dictionaries to a CSV file with a dynamic prefix."""
    
    filename = f"{combination_name}-token_results.csv"
    fieldnames = list(results[0].keys())
    
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(results)
        
    print(f"Successfully saved {len(results)} rows to {filename}")

def load_results_from_csv(combination_name):
    """Loads token results from a CSV file with a dynamic prefix back into a list."""
    filename = f"{combination_name}-token_results.csv"
    loaded_results = []
    
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        
        for row in reader:
            loaded_results.append({
                "token_id": int(row["token_id"]),
                "token_str": row["token_str"],
                "appearances": int(row["appearances"]),
                "avg_base_logprob": float(row["avg_base_logprob"]),
                "avg_steer_logprob": float(row["avg_steer_logprob"]),
                "logprob_shift": float(row["logprob_shift"])
            })
            
    print(f"Successfully loaded {len(loaded_results)} rows from {filename}")
    return loaded_results

### Running examples

#### 1. Boxing + Thailand

In [ ]:
concepts = ["Boxing", "Thailand"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Boxing", "Thailand"],
    neg_prompts= [" ", " "],
    hypotheses= ["Boxing", "Thailand","Muay Thai", "Combat"],
    layers= [1,2],
    coeffs= [3,3],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 2,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 2. Sports + Winter

In [ ]:
concepts = ["Sports","Winter"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Sports", "I like winter"],
    neg_prompts= [" ", "I like summer"],
    hypotheses= ["Sports", "Winter", "Winter Sports"],
    layers= [1,2],
    coeffs= [3,3],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 3. The Sun + Eyeglasses

In [ ]:
concepts = ["The Sun","Eyeglasses"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["the sun", "eyeglasses"],
    neg_prompts= [" ", " "],
    hypotheses= ["The Sun", "Eyeglasses", "Sunglasses"],
    layers= [1,2],
    coeffs= [4,3],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 1,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 4. Horse + Racing

In [ ]:
concepts = ["Horse","Racing"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Horse", "Racing"],
    neg_prompts= [" ", " "],
    hypotheses= ["Horse", "Racing", "Horse Racing"],
    layers= [3,5],
    coeffs= [2.5,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 2,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 5. Polar + Bear

In [ ]:
concepts = ["Polar","Bear"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Polar", "Bear"],
    neg_prompts= [" ", " "],
    hypotheses= ["Polar", "Bear", "Polar Bear"],
    layers= [3,4],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 6. Value + Proposition

In [ ]:
concepts = ["Value","Proposition"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Value", "Proposition"],
    neg_prompts= [" ", " "],
    hypotheses= ["Value", "Proposition", "Value Proposition"],
    layers= [3,4],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 2,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 7. Toilet + Paper

In [ ]:
concepts = ["Toilet","Paper"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Toilet", "Paper"],
    neg_prompts= [" ", " "],
    hypotheses= ["Toilet","Paper","Toilet Paper"],
    layers= [3,1],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 8. Japanese + Cartoon

In [ ]:
concepts = ["Japanese","Cartoon"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Japanese","Cartoon"],
    neg_prompts= [" ", " "],
    hypotheses= ["Cartoon","Japanese","Anime"],
    layers= [1,2],
    coeffs= [10,3],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 9. Pacifist vs Murderous

In [ ]:
concepts = ["Pacifist","Murderous"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I am a comitted pacifist", "I am murderous"],
    neg_prompts= [" ", " "],
    hypotheses= ["Pacifism","Violent Crimes","Murder"],
    layers= [10,30],
    coeffs= [7,7],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 10. Rich vs Poor

In [ ]:
concepts = ["Rich","Poor"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I am rich", "I am poor"],
    neg_prompts= [" ", " "],
    hypotheses= ["Being Rich","Being Poor"],
    layers= [3,4],
    coeffs= [1.5,1.5],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 11. Desert + Rainforest

In [ ]:
concepts = ["Desert","Rainforest"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Desert", "Rainforest"],
    neg_prompts= [" ", " "],
    hypotheses= ["Desert","Rainforest","Forest"],
    layers= [2,4],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 12. Man vs Woman

In [ ]:
concepts = ["I talk like a man","I talk like a woman"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I talk like a man", "I talk like a woman"],
    neg_prompts= ["I talk like a woman", "I talk like a man"],
    hypotheses= ["Gender"],
    layers= [9,3],
    coeffs= [9,9],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 13. Winner vs Loser

In [ ]:
concepts = ["I talk like a winner","I talk like a loser"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I talk like a winner", "I talk like a loser"],
    neg_prompts= ["I talk like a loser", "I talk like a winner"],
    hypotheses= ["I talk like a winner","I talk like a loser"],
    layers= [5,4],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts) + '1')

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 14. Winner vs loser

In [ ]:
concepts = ["I talk like a winner","I talk like a loser"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I talk like a winner", "I talk like a loser"],
    neg_prompts= ["I do not talk like a winner", "I do not talk like a loser"],
    hypotheses= ["I talk like a winner","I talk like a loser"],
    layers= [5,4],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts) + '2')

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 15. Cold vs Warm

In [ ]:
concepts = ["Cold","Warm"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is cold in here", "It is warm in here"],
    neg_prompts= ["It is warm in here", "It is cold in here"],
    hypotheses= ["Clothes","Temperature","Love"],
    layers= [1,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 16. Cold vs Warm

In [ ]:
concepts = ["Cold","Warm"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is cold in here", "It is warm in here"],
    neg_prompts= ["It is not cold in here", "It is not warm in here"],
    hypotheses= ["Clothes","Temperature","Love"],
    layers= [1,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 17. Vegeterian vs Meat

In [ ]:
concepts = ["Vegetarian","Meat-eater"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I am a vegetarian", "I love eating meat"],
    neg_prompts= ["I am not a vegetarian", "I hate eating meat"],
    hypotheses= ["Vegan","Meat","Food"],
    layers= [1,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 18. Vegeterian vs Meat

In [ ]:
concepts = ["Vegetarian","Meat-eater"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["I am a vegetarian", "I love eating meat"],
    neg_prompts= ["I love eating meat", "I am a vegetarian"],
    hypotheses= ["Vegan","Meat","Food"],
    layers= [1,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 19. Winter vs Summer

In [ ]:
concepts = ["Winter","Summer"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is winter", "It is summer"],
    neg_prompts= ["It is not winter", "It is not summer"],
    hypotheses= ["Summertime","Wintertime","Summer Activities"],
    layers= [5,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)


#### 20. Winter vs Summer

In [ ]:
concepts = ["Winter","Summer"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is winter", "It is summer"],
    neg_prompts= ["It is summer", "It is winter"],
    hypotheses= ["Summertime","Wintertime","Summer Activities"],
    layers= [5,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)


#### 21. Day vs Night

In [ ]:
concepts = ["Day", "Night"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is daytime","It is nighttime"],
    neg_prompts= ["It is not daytime", "It is not nighttime"],
    hypotheses=["Night","Morning"],
    layers= [6,8],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)

#### 22. Day vs Night

In [ ]:
concepts = ["Day", "Night"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["It is daytime","It is nighttime"],
    neg_prompts= ["It is nighttime", "It is daytime"],
    hypotheses=["Night","Morning"],
    layers= [6,8],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)


#### 23. Basketball vs Not Basketball

In [ ]:
concepts = ["Basketball","Not Basketball"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Basketball"," "],
    neg_prompts= [" ","Basketball"],
    hypotheses=["Basketball"],
    layers= [1,3],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)



#### 24. Restaurant vs Not Restaurant

In [ ]:
concepts = ["Restaurant", "Not Restaurant"]
keywords_csv_path="data_hub\log_prob_shift_keywords.csv"
results = get_logprob_shifts(
    concept_names= concepts,
    pos_prompts= ["Restaurant"," "],
    neg_prompts= [" ", "Restaurant"],
    hypotheses=["Restaurant"],
    layers= [3,5],
    coeffs= [2,2],
    num_docs = 500,
    batch_size= 4,
    token_min_appearances = 10,
    max_length= 1024,
    min_doc_keywords= 3,
    keywords_csv_path=keywords_csv_path,
)

# save results in a file
save_results_to_csv(results, '+'.join(concepts))

plot_shift_distribution(results, concepts )
print_top_and_bottom_shifts(results, concepts, 50, 50 )
plot_qq_shifts(results, concepts)
